**Модуль 1. Введение: почему инфраструктура важнее кода**

### 1.1. Что происходит после обучения модели

**Контекст для новичка.**  
Когда вы впервые обучаете модель машинного обучения, вы обычно работаете в Jupyter Notebook или Google Colab. Вы загружаете датасет, чистите данные, запускаете `model.fit()`, смотрите на метрики — и радуетесь. На этом этапе важен *эксперимент*: быстро проверить гипотезу, подобрать параметры, увидеть, что модель вообще способна давать полезный результат.

Но в реальном мире модель не существует ради самой модели. Она существует ради *пользователя*, который хочет получить от неё пользу: предсказание цены квартиры, распознавание текста на фото, рекомендацию фильма. Чтобы пользователь получил это предсказание, модель должна быть:

1. **Доступна** — работать 24/7 на каком-то компьютере (сервере), а не на вашем ноутбуке.
2. **Предсказуема** — давать одинаковый результат при одинаковых входных данных.
3. **Масштабируема** — выдерживать нагрузку: одновременно могут обращаться 10, 1000 или миллион пользователей.
4. **Отказоустойчива** — если один компонент сломался, система должна продолжать работать или корректно восстановиться.

Всё это — **инфраструктура**. Код модели — это лишь небольшая часть огромной системы. В индустрии говорят: *«Любой может написать модель в ноутбуке. Но запустить её в production — вот где начинается настоящая работа».*

### 1.2. Проблема «работает на моём ноутбуке»

**Что такое «окружение»?**  
Программа — это не только код, который вы написали. Это ещё и:

- **Операционная система** (Windows, macOS, Linux).
- **Версия языка программирования** (Python 3.9, 3.11, 3.12).
- **Версии библиотек** (`numpy==1.24` ведёт себя иначе, чем `numpy==2.0`).
- **Системные зависимости** (компиляторы C/C++, которые ставятся отдельно от Python).
- **Переменные окружения** (настройки, которые хранятся в операционной системе, а не в коде).

**Аналогия.**  
Представьте, что вы написали рецепт торта. Вы готовили его на своей кухне, где в шкафчике лежит мука определённой марки, духовка нагревается до 180°C, а яйца среднего размера. Вы даёте рецепт другу. У него другая мука, духовка греет на 200°C, а яйца крупные. Торт получится другим. Может быть, он вообще не поднимется.

То же самое с кодом. Вы написали скрипт на macOS с Python 3.11 и `scikit-learn==1.3`. Ваш коллега запускает его на сервере с Linux и Python 3.9. Скрипт падает с ошибкой. Или, что хуже, работает, но даёт другие предсказания, потому что внутри библиотеки изменилась логика округления чисел.

**Почему это критично для ML?**  
В машинном обучении воспроизводимость — основа доверия. Если модель показала accuracy 95% на вашем ноутбуке, но 89% на сервере, вы не можете быть уверены, что деплоите правильную версию. Инфраструктура решает эту проблему, гарантируя: *«Где бы ни запускался код, окружение будет абсолютно одинаковым».*

### 1.3. Зачем отделять «быстрые» запросы от «медленных» задач

**Что такое API (простыми словами)?**  
API (Application Programming Interface) — это точка входа в вашу систему извне. Представьте его как окошко в ресторане быстрого питания: пользователь подходит, делает заказ (HTTP-запрос), и ждёт ответ. Если заказ готовится 2 секунды — отлично. Если 5 минут — пользователь уйдёт.

**Две скорости в ML-системах:**

1. **Быстрые задачи (синхронные).**  
   Пользователь загружает фото, API возвращает предсказание модели за 200 миллисекунд. Пользователь ждёт прямо здесь и сейчас. Это *онлайн-инференс*.

2. **Медленные задачи (асинхронные).**  
   Пользователь загружает 10 гигабайт данных для обучения новой модели. Обработка займёт 30 минут. Нельзя заставлять пользователя сидеть и ждать у экрана полчаса, глядя на крутящийся спиннер.

**Аналогия с рестораном.**  
Представьте ресторан, где официант (API) принимает заказы. Если клиент заказывает кофе — бариста готовит его за минуту, официант стоит и ждёт, затем несёт. Это быстрый запрос.  
Но если клиент заказывает банкет на 50 человек — официант не будет стоять у плиты 3 часа. Он запишет заказ на листочек (поставит задачу в очередь), отдаст его на кухню, и скажет клиенту: *«Ваш заказ №42 принят. Мы сообщим, когда будет готов».* Затем он свободен принимать следующих клиентов.

**Зачем это нужно?**  
- **Отзывчивость.** API мгновенно отвечает пользователю, даже если задача тяжёлая.
- **Надёжность.** Если сервер, выполняющий тяжёлую задачу, упал — заказ не пропадает, он остаётся в очереди.
- **Масштабируемость.** Можно добавить 10 поваров (воркеров) на кухню, не меняя окошко заказа (API).

### 1.4. Общая картина курса: путь от ноутбука до production

В этом курсе мы пройдём путь, который проходит любая ML-система при выходе в реальный мир. Вот карта:

**Этап 1. Упаковка — Docker.**  
Мы научимся создавать «контейнеры» — лёгкие, изолированные виртуальные коробки, в которых живут ваш код, Python, библиотеки и даже операционная система. Контейнер решает проблему «работает на моём ноутбуке»: вы собираете его один раз, и он работает одинаково на вашем компьютере, на сервере коллеги и в облаке Amazon.

**Этап 2. Связка сервисов — Docker Compose.**  
Реальная система состоит не из одного контейнера, а из многих: API на Python, база данных PostgreSQL, брокер сообщений Redis. Мы научимся описывать всю эту экосистему в одном файле `docker-compose.yml` и запускать её одной командой.

**Этап 3. Очереди задач — Celery + Redis.**  
Мы научимся отдавать тяжёлые задачи (обучение, предобработка) в фон, используя паттерн Producer/Consumer. API быстро принимает заказ, а воркеры в фоновом режиме его выполняют.

**Этап 4. Потоки событий — Kafka.**  
Когда система вырастает, нужно не просто выполнять задачи, а *фиксировать факты*: «пользователь загрузил файл», «модель обучена», «предсказание сделано». Kafka — это распределённый журнал событий, который позволяет разным частям системы общаться надёжно и в реальном времени.

### 1.5. Практика: ваша первая встреча с Docker

Цель этого практического блока — убедиться, что Docker установлен и работает, и увидеть контейнер «вживую». Не пытайтесь понять каждую деталь сейчас; мы разберём их позже. Просто выполните шаги и зафиксируйте ощущение.

**Шаг 1. Установка Docker Desktop.**  
Скачайте и установите Docker Desktop для вашей операционной системы (Windows, macOS или Linux) с сайта [docker.com](https://www.docker.com). Это приложение, которое управляет контейнерами на вашем компьютере.

**Шаг 2. Проверка установки.**  
Откройте терминал (командную строку) и выполните:

In [ ]:
docker --version

Вы должны увидеть что-то вроде:

In [ ]:
Docker version 27.1.1, build 6312585

Это означает, что Docker установлен и готов к работе.

**Шаг 3. Запуск первого контейнера.**  
Выполните команду:

In [ ]:
docker run hello-world

**Что произошло?**  
1. Docker проверил, есть ли образ `hello-world` на вашем компьютере. Его не было.
2. Docker скачал образ из интернета (из публичного реестра Docker Hub).
3. Docker создал из образа *контейнер* — запущенный экземпляр.
4. Контейнер выполнил маленькую программу, которая напечатала приветственное сообщение и объяснила, что такое Docker.
5. Контейнер завершил работу и остановился.

**Что важно запомнить:**  
- **Образ (image)** — это шаблон, «фотография» системы (как ISO-диск).
- **Контейнер (container)** — это запущенный экземпляр этого образа (как запущенная виртуальная машина, но гораздо легче).
- Один образ можно запустить в десятки контейнеров одновременно.

**Шаг 4. Посмотрите на список контейнеров.**  
Выполните:

In [ ]:
docker ps -a

Вы увидите таблицу со столбцами `CONTAINER ID`, `IMAGE`, `STATUS`. Ваш `hello-world` будет в статусе `Exited` — он отработал и остановился. Это нормально: контейнер выполнил свою задачу и завершился.

### 1.6. Итоги модуля: что мы узнали

- **ML в production** — это не только модель, но и инфраструктура вокруг неё.
- **Проблема окружения** реальна: разные ОС, версии Python и библиотек ломают воспроизводимость.
- **Быстрые и медленные задачи** нужно разделять, чтобы API оставался отзывчивым.
- **Docker** решает проблему окружения, упаковывая приложение в изолированный контейнер.
- Мы уже запустили первый контейнер и увидели, как образ превращается в работающую программу.

**В следующем модуле** мы начнём разбирать Docker глубже: узнаем, как устроена изоляция, и научимся запускать готовые образы Python, чтобы выполнять код внутри контейнера.